# pcode2cpp · one function at a time

Every cell is manual. No tools or retry loops inside Inkling. Run from the repository root (`jupyter lab`). `config.json` and all cases remain untracked.


In [ ]:
from pathlib import Path
import json
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from pcode2cpp.ghidra import export_function
from pcode2cpp.translate import build_messages, translate
from integrations.spore_decomp import SporeDecompBackend

CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
ADDRESS = 0x00672CF0
CASE = ROOT / "cases" / f"{ADDRESS:08x}"
CASE.mkdir(parents=True, exist_ok=True)
print("Target:", hex(ADDRESS), "Output:", CASE)


## 1. Export Ghidra context

Uses the existing analyzed Ghidra project. PDB information must already be imported into that project. Re-running this cell exports afresh.


In [ ]:
context = export_function(ADDRESS, CASE / "context.json", CONFIG["ghidra"])
print("Function:", context["function"]["name"])
print("P-code ops:", len(context["high_pcode"]), "Assembly instructions:", len(context["assembly"]))
print("Calls:", len(context["calls"]), "Referenced types:", len(context["types"]))


## 2. Inspect what Ghidra found

Review function identity, every call target, type provenance, and pseudocode before inference. The complete context is saved in `context.json`.


In [ ]:
print(json.dumps(context["function"], indent=2))
print("\nCALLS:\n", json.dumps(context["calls"], indent=2))
print("\nTYPES:\n", json.dumps(context["types"], indent=2)[:12000])
print("\nDECOMPILED C:\n", context["pseudocode"])
print("\nHIGH PCODE (first 20 for display only):\n", json.dumps(context["high_pcode"][:20], indent=2))


## 3. Inspect the entire prompt

The complete JSON, not the displayed previews above, is sent to Inkling. The model context window must accommodate the entire request and expected output. Nothing is truncated by pcode2cpp.


In [ ]:
messages = build_messages(context)
print("SYSTEM INSTRUCTION:\n", messages[0]["content"])
print("\nFULL USER CONTEXT:\n", messages[1]["content"])
print("\nInput characters:", sum(len(m["content"]) for m in messages))


## 4. Translate once using local Inkling

This cell makes exactly one request to the local llama.cpp server and saves raw C++. No tools, retries, or OpenRouter.


In [ ]:
cpp, metadata = translate(context, CONFIG["llm"])
(CASE / "candidate.cpp").write_text(cpp, encoding="utf-8")
(CASE / "inference.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(cpp)
print("\nInference metadata:", metadata)


## 5. Optional Spore verification

**Manual preparation:** Fit `cases/<address>/candidate.cpp` into the correct Spore compilation unit in a dedicated Spore checkout, preserving the `// @ 0x...` marker. This cell never copies or overwrites sources. `check.py --dry-run` performs a byte match, then `prove.py` attempts equivalence if needed. Use a separate checkout so your normal worktree is untouched.


In [ ]:
SPORE_SOURCE = None  # e.g. Path(CONFIG["spore"]["repo_path"]) / "match/slices/.../...cpp"
if SPORE_SOURCE is None:
    print("Skipped: set SPORE_SOURCE to the prepared translation unit to verify.")
else:
    spore = CONFIG["spore"]
    backend = SporeDecompBackend(Path(spore["repo_path"]), spore.get("python", ".venv/bin/python"), spore.get("timeout_seconds", 600))
    result = backend.verify(ADDRESS, Path(SPORE_SOURCE))
    (CASE / "verification.txt").write_text(result.verdict.value + "\n" + result.details + "\n" + result.compiler_output, encoding="utf-8")
    print("Verdict:", result.verdict.value, "Proof layer:", result.proof_method)
    print(result.details)
